# 00 - Ollama y embeddings con el cliente oficial

Objetivo: usar `Ollama` por separado para dos tareas distintas:

- generar texto con un modelo de chat,
- generar embeddings con el modelo de vectores.

Esta version usa el paquete oficial `ollama` directamente, sin wrappers externos y sin construir llamadas HTTP manuales.

In [1]:
import os
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from ollama import Client

load_dotenv(Path("..").resolve() / ".env")

OLLAMA_PORT = os.getenv("OLLAMA_PORT", "11434")
CHAT_MODEL = os.getenv("CHAT_MODEL", "llama3")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "embeddinggemma")
OLLAMA_BASE_URL = f"http://127.0.0.1:{OLLAMA_PORT}"

client = Client(host=OLLAMA_BASE_URL)

print({
    "OLLAMA_BASE_URL": OLLAMA_BASE_URL,
    "CHAT_MODEL": CHAT_MODEL,
    "EMBEDDING_MODEL": EMBEDDING_MODEL,
})


{'OLLAMA_BASE_URL': 'http://127.0.0.1:11434', 'CHAT_MODEL': 'llama3', 'EMBEDDING_MODEL': 'embeddinggemma'}


## Hacer `pull` de un modelo desde Python

Si el modelo todavia no existe en Ollama, podemos descargarlo desde codigo usando el metodo `pull` del cliente oficial.

In [2]:
def pull_model(model_name: str) -> dict:
    result = client.pull(model=model_name, stream=False)
    return result.model_dump(exclude_none=True)

# Ejemplo: asegura que el modelo de embeddings este disponible.
# Puedes cambiarlo por CHAT_MODEL si quieres descargar el modelo de chat.
result = pull_model(EMBEDDING_MODEL)
pprint(result)


{'status': 'success'}


## Generar texto

`client.chat` recibe una lista de mensajes y devuelve una respuesta con el contenido en `response["message"]["content"]`.

In [3]:
response = client.chat(
    model=CHAT_MODEL,
    messages=[
        {
            "role": "user",
            "content": "Resume en dos frases para que sirve un sistema RAG.",
        }
    ],
    options={"temperature": 0},
)

print(response["message"]["content"])


Un sistema de seguimiento de problemas (RAG) es una herramienta utilizada para clasificar y priorizar problemas o proyectos en tres categorías: Rojo (prioridad alta, problema crítico), Amarillo (prioridad media, problema importante) y Verde (prioridad baja, problema resuelto). El objetivo es facilitar la toma de decisiones y la asignación de recursos efectivos para abordar los problemas más urgentes.


## Generar embeddings

`client.embed` permite enviar una cadena o una lista de cadenas. En este caso devuelve una lista de vectores en `response["embeddings"]`.

In [5]:
sentences = [
    "La politica de devoluciones permite cambiar un producto dentro de plazo.",
    "El manual de montaje explica los tornillos y herramientas necesarios.",
    "Qdrant almacena vectores y payload para recuperar contexto relevante.",
]

embedding_response = client.embed(model=EMBEDDING_MODEL, input=sentences)
vectors = embedding_response["embeddings"]

print(f"Numero de textos: {len(vectors)}")
print(f"Dimension del embedding: {len(vectors[0])}")
print("Primeros 8 valores del primer vector:")
pprint(vectors[0][:8])


Numero de textos: 3
Dimension del embedding: 768
Primeros 8 valores del primer vector:
[-0.07008405,
 -0.050034113,
 0.00072373415,
 0.010708857,
 -0.0140296295,
 0.05033357,
 -0.018281631,
 0.06515346]
